# Divye FE — Joint Target Encoding

Extension of `s6e2-divye-fe.ipynb`. Instead of multiplying two marginal TE values  
(which assumes independence), compute the **joint TE** of feature pairs directly:

- Group rows by `(thallium, chest_pain_type)` together
- Compute mean target rate for that exact combination
- This captures the true joint probability, not the product approximation

**Example**: `thallium=7, chest_pain=4` group might have 91% disease rate —  
but `thallium_te × chest_pain_te = 0.82 × 0.76 = 0.62`, which understimates the joint signal.

Applies to top-8 TE features (same selection as base notebook).  
Numeric features are binned into deciles before joint TE to keep group sizes manageable.  
Baseline: catboost_divye_fe OOF = 0.95566

In [1]:
import subprocess
import numpy as np
import pandas as pd
from itertools import combinations
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(train)
test  = prep(test)

CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
NUM_FEATURES = ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']
ALL_FEATURES = CAT_FEATURES + NUM_FEATURES

X      = train[ALL_FEATURES]
y      = train['heart_disease'].values
X_test = test[ALL_FEATURES]

SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'Train: {X.shape}  Test: {X_test.shape}')

Train: (630000, 13)  Test: (270000, 13)


In [2]:
def compute_freq_features(train_df, test_df, cols):
    combined = pd.concat([train_df[cols], test_df[cols]])
    tr_freq, te_freq = {}, {}
    for col in cols:
        freq_map = combined[col].value_counts(normalize=True)
        tr_freq[f'{col}_freq'] = train_df[col].map(freq_map).fillna(0).values
        te_freq[f'{col}_freq'] = test_df[col].map(freq_map).fillna(0).values
    return tr_freq, te_freq


def compute_oof_te(train_df, test_df, cols, y, skf):
    global_mean = y.mean()
    oof_te  = {f'{col}_te': np.zeros(len(train_df)) for col in cols}
    test_te = {f'{col}_te': np.zeros(len(test_df))  for col in cols}
    for col in cols:
        key, tr_vals, te_vals, fold_test = f'{col}_te', train_df[col].values, test_df[col].values, []
        for tr_idx, val_idx in skf.split(train_df, y):
            te_map = pd.DataFrame({'v': tr_vals[tr_idx], 't': y[tr_idx]}).groupby('v')['t'].mean()
            oof_te[key][val_idx] = pd.Series(tr_vals[val_idx]).map(te_map).fillna(global_mean).values
            fold_test.append(pd.Series(te_vals).map(te_map).fillna(global_mean).values)
        test_te[key] = np.mean(fold_test, axis=0)
    return oof_te, test_te


def bin_numeric(train_df, test_df, num_cols, n_bins=10):
    """Replace numeric columns with decile bin labels (strings) for joint TE."""
    tr, te = train_df.copy(), test_df.copy()
    for col in num_cols:
        if col in train_df.columns:
            _, edges = pd.qcut(train_df[col], q=n_bins, retbins=True, duplicates='drop')
            tr[col] = pd.cut(train_df[col], bins=edges, labels=False, include_lowest=True).astype(str)
            te[col] = pd.cut(test_df[col],  bins=edges, labels=False, include_lowest=True).astype(str).fillna('unk')
    return tr, te


def compute_oof_joint_te(train_df, test_df, col_pairs, y, skf):
    """OOF TE for feature pairs. Uses concatenated string key per pair."""
    global_mean = y.mean()
    oof_jte  = {}
    test_jte = {}

    for c1, c2 in col_pairs:
        key = f'{c1}_joint_{c2}_te'
        tr_combo = train_df[c1].astype(str) + '_' + train_df[c2].astype(str)
        te_combo = test_df[c1].astype(str)  + '_' + test_df[c2].astype(str)
        tr_vals, te_vals = tr_combo.values, te_combo.values

        oof_arr, fold_test = np.zeros(len(train_df)), []
        for tr_idx, val_idx in skf.split(train_df, y):
            te_map = pd.DataFrame({'v': tr_vals[tr_idx], 't': y[tr_idx]}).groupby('v')['t'].mean()
            oof_arr[val_idx] = pd.Series(tr_vals[val_idx]).map(te_map).fillna(global_mean).values
            fold_test.append(pd.Series(te_vals).map(te_map).fillna(global_mean).values)

        oof_jte[key]  = oof_arr
        test_jte[key] = np.mean(fold_test, axis=0)

    return oof_jte, test_jte


def build_augmented(base_df, *dicts):
    df = base_df.copy().reset_index(drop=True)
    for d in dicts:
        for name, vals in d.items():
            df[name] = vals
    return df


# Precompute on full data
print('Frequency encoding...')
tr_freq, te_freq = compute_freq_features(X, X_test, ALL_FEATURES)

print('OOF target encoding (individual features)...')
oof_te, test_te = compute_oof_te(X, X_test, ALL_FEATURES, y, SKF)

# Select top-8 by individual TE correlation (same as base notebook)
corrs = {col: abs(np.corrcoef(vals, y)[0, 1]) for col, vals in oof_te.items()}
top8_keys = sorted(corrs, key=corrs.get, reverse=True)[:8]  # e.g. 'thallium_te'
top8_cols = [k.replace('_te', '') for k in top8_keys]       # strip _te suffix for original col names
print(f'Top-8 features: {top8_cols}')

# Bin numeric features that are in top-8 (for joint TE)
num_in_top8 = [c for c in top8_cols if c in NUM_FEATURES]
print(f'Numeric in top-8 (will be binned for joint TE): {num_in_top8}')
X_binned,      X_test_binned      = bin_numeric(X,      X_test, num_in_top8)

# All C(8,2) = 28 pairs
top8_pairs = list(combinations(top8_cols, 2))
print(f'\nComputing {len(top8_pairs)} joint TE pairs...')
oof_jte, test_jte = compute_oof_joint_te(X_binned, X_test_binned, top8_pairs, y, SKF)
print(f'Joint TE features: {len(oof_jte)}')

Frequency encoding...
OOF target encoding (individual features)...
Top-8 features: ['thallium', 'chest_pain_type', 'max_hr', 'number_of_vessels_fluro', 'st_depression', 'exercise_angina', 'slope_of_st', 'sex']
Numeric in top-8 (will be binned for joint TE): ['max_hr', 'st_depression']

Computing 28 joint TE pairs...
Joint TE features: 28


In [3]:
CATBOOST_PARAMS = dict(
    iterations=2000, learning_rate=0.05, depth=6,
    task_type='CPU', thread_count=-1,
    cat_features=CAT_FEATURES, random_seed=42, verbose=0,
)

global_mean = y.mean()
oof_preds   = np.zeros(len(y))
test_folds  = np.zeros((5, len(X_test)))

for fold, (tr_idx, val_idx) in enumerate(SKF.split(X, y)):
    print(f'\n=== Fold {fold + 1}/5 ===')
    X_tr_base  = X.iloc[tr_idx].reset_index(drop=True)
    X_val_base = X.iloc[val_idx].reset_index(drop=True)
    y_tr, y_val = y[tr_idx], y[val_idx]

    # Freq
    fold_tr_freq, fold_te_freq  = compute_freq_features(X_tr_base, X_test,    ALL_FEATURES)
    _,            fold_val_freq = compute_freq_features(X_tr_base, X_val_base, ALL_FEATURES)

    # Individual TE per fold
    fold_tr_te, fold_val_te, fold_te_te = {}, {}, {}
    for col in ALL_FEATURES:
        key    = f'{col}_te'
        te_map = pd.DataFrame({'v': X_tr_base[col].values, 't': y_tr}).groupby('v')['t'].mean()
        fold_tr_te[key]  = X_tr_base[col].map(te_map).fillna(global_mean).values
        fold_val_te[key] = X_val_base[col].map(te_map).fillna(global_mean).values
        fold_te_te[key]  = X_test[col].map(te_map).fillna(global_mean).values

    # Bin numerics for joint TE
    X_tr_bin, _          = bin_numeric(X_tr_base, X_test,    num_in_top8)
    X_val_bin_tr, _      = bin_numeric(X_tr_base, X_val_base, num_in_top8)
    X_val_bin = X_val_bin_tr  # reuse edges from training fold
    _, X_te_bin = bin_numeric(X_tr_base, X_test, num_in_top8)

    # Joint TE per fold
    fold_tr_jte, fold_val_jte, fold_te_jte = {}, {}, {}
    for c1, c2 in top8_pairs:
        key     = f'{c1}_joint_{c2}_te'
        tr_combo  = X_tr_bin[c1].astype(str) + '_' + X_tr_bin[c2].astype(str)
        val_combo = X_val_base[c1].astype(str) + '_' + X_val_base[c2].astype(str)
        te_combo  = X_te_bin[c1].astype(str) + '_' + X_te_bin[c2].astype(str)
        jte_map = pd.DataFrame({'v': tr_combo.values, 't': y_tr}).groupby('v')['t'].mean()
        fold_tr_jte[key]  = tr_combo.map(jte_map).fillna(global_mean).values
        fold_val_jte[key] = val_combo.map(jte_map).fillna(global_mean).values
        fold_te_jte[key]  = te_combo.map(jte_map).fillna(global_mean).values

    X_tr_aug  = build_augmented(X_tr_base,  fold_tr_freq,  fold_tr_te,  fold_tr_jte)
    X_val_aug = build_augmented(X_val_base, fold_val_freq, fold_val_te, fold_val_jte)
    X_te_aug  = build_augmented(X_test,     fold_te_freq,  fold_te_te,  fold_te_jte)

    m = CatBoostClassifier(**CATBOOST_PARAMS)
    m.fit(X_tr_aug, y_tr, eval_set=(X_val_aug, y_val), early_stopping_rounds=50)

    oof_preds[val_idx] = m.predict_proba(X_val_aug)[:, 1]
    test_folds[fold]   = m.predict_proba(X_te_aug)[:, 1]
    print(f'Fold {fold+1}  AUC={roc_auc_score(y_val, oof_preds[val_idx]):.5f}  best_iter={m.best_iteration_}')

cv_auc     = roc_auc_score(y, oof_preds)
test_preds = test_folds.mean(axis=0)

print(f'\nOOF AUC (joint_te): {cv_auc:.5f}')
print(f'Baseline divye_fe:  0.95566')
print(f'Delta:              {cv_auc - 0.95566:+.5f}')


=== Fold 1/5 ===
Fold 1  AUC=0.95338  best_iter=491

=== Fold 2/5 ===
Fold 2  AUC=0.95276  best_iter=1068

=== Fold 3/5 ===
Fold 3  AUC=0.95292  best_iter=908

=== Fold 4/5 ===
Fold 4  AUC=0.95291  best_iter=1111

=== Fold 5/5 ===
Fold 5  AUC=0.95349  best_iter=907

OOF AUC (joint_te): 0.95295
Baseline divye_fe:  0.95566
Delta:              -0.00271


In [ ]:
np.save('submissions/oof_divye_fe_joint.npy', oof_preds)

label = 'catboost_divye_fe_joint'
fname = f'submissions/{label}.csv'
sub = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)

desc = f'{label} | cv_auc={cv_auc:.4f}'
print(f'Saved: {fname}')
print(f'desc:  {desc}')
print(f'OOF AUC: {cv_auc:.5f}  (baseline divye_fe: 0.95566  delta: {cv_auc - 0.95566:+.5f})')

In [ ]:
import subprocess
r = subprocess.run(
    ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
     '-f', fname, '-m', desc],
    capture_output=True, text=True
)
status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
print(f'{label}: {status}')